<div  style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Associate/main/Includes/images/bookstore_schema.png" alt="Databricks Learning" style="width: 600">
</div>

In [0]:
%run ../Include/Copy-Datasets

In [0]:
%python
(spark.readStream
      .table("books_csv")
      .createOrReplaceTempView("books_streaming_tmp_vw")
)    
 


In [0]:
%sql
SELECT * FROM books_streaming_tmp_vw


In [0]:
(spark.readStream
      .table("books_csv")
      .writeStream
      .format("memory")
      .queryName("books_streaming_tmp_vw")
      .option("checkpointLocation", "abfss://unity-catalog-storage@dbstorageb5xzbjmlfqwie.dfs.core.windows.net/7405611205197801/checkpoints/books_streaming")
      .start()

In [0]:
%sql
SELECT * FROM books_streaming_tmp_vw

In [0]:
%python
df = spark.sql("""
    SELECT author, count(book_id) AS total_books
    FROM books_streaming_tmp_vw
    GROUP BY author
""")

display(df, checkpointLocation = "abfss://unity-catalog-storage@dbstorageb5xzbjmlfqwie.dfs.core.windows.net/7405611205197801/checkpoints/books_streaming_agg")

In [0]:
df = spark.sql("""
    SELECT *
    FROM books_streaming_tmp_vw
    ORDER BY author
""")

display(df)


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW author_counts_tmp_vw AS (
    SELECT author, count(book_id) AS total_books
    FROM books_streaming_tmp_vw
    GROUP BY author
)

In [0]:
(spark.table("author_counts_tmp_vw")
      .writeStream
      .trigger(processingTime='4 seconds')
      .outputMode("complete")
      .option("checkpointLocation", "abfss://unity-catalog-storage@dbstorageb5xzbjmlfqwie.dfs.core.windows.net/7405611205197801/checkpoints/author_counts")
      .table("author_counts")
)


In [0]:
%sql
SELECT * FROM author_counts

In [0]:
%sql
INSERT INTO books_csv 
values ("B19", "Introduction to Modeling and Simulation", "Mark W. Spong", "Computer Science", 25),
        ("B20", "Robot Modeling and Control", "Mark W. Spong", "Computer Science", 30),
        ("B21", "Turing's Vision: The Birth of Computer Science", "Chris Bernhardt", "Computer Science", 35)

In [0]:
%sql
SELECT * FROM author_counts


In [0]:
%sql
INSERT INTO books_csv 
values ("B16", "Hands-On Deep Learning Algorithms with Python", "Sudharsan Ravichandiran", "Computer Science", 25),
        ("B17", "Neural Network Methods in Natural Language Processing", "Yoav Goldberg", "Computer Science", 30),
        ("B18", "Understanding digital signal processing", "Richard Lyons", "Computer Science", 35)

In [0]:
(spark.table("author_counts_tmp_vw")
      .writeStream
      .trigger(availableNow=True)
      .outputMode("complete")
      .option("checkpointLocation", "abfss://unity-catalog-storage@dbstorageb5xzbjmlfqwie.dfs.core.windows.net/7405611205197801/checkpoints/author_counts")
      .table("author_counts")
      .awaitTermination()
)


In [0]:
%sql
SELECT * FROM author_counts